In [ ]:
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install fastapi uvicorn pyngrok nest-asyncio pydantic
!pip install transformers accelerate
!pip install openai-whisper librosa

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import librosa
import io
import os
import whisper

device = "cuda" if torch.cuda.is_available() else "cpu"

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

In [ ]:
ASRka = whisper.load_model("turbo", device=device)

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):
    audio_content = await file.read()
    temp_file = "temp_voice.webm"
    with open(temp_file, "wb") as f:
        f.write(audio_content)
    try:
        result = ASRka.transcribe(temp_file, fp16=(device == "cuda"))
        return {"text": result["text"].strip()}
    except Exception as e:
        print(f"ASR Error: {e}")
        return {"text": "Error transcribing audio"}
    finally:
        if os.path.exists(temp_file):
            os.remove(temp_file)

In [ ]:
from pydantic import BaseModel
class Question(BaseModel):
    question: str

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
LLMka = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16, # Use float16 for speed
    device_map="auto"
)

@app.post("/ask")
async def ask(q: Question):
    prompt = f"<|system|>\nYou are a helpful assistant.</s>\n<|user|>\n{q.question}</s>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(LLMka.device)
    with torch.no_grad():
        outputs = LLMka.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

    return {"answer": response.strip()}

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import asyncio

SHOULD_IT_BE_HIDDEN = "..."

ngrok.set_auth_token(SHOULD_IT_BE_HIDDEN)

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("PUBLIC URL:", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()